# Matching field polytopes for $\mathrm{Gr}(3,6)$

Let's compare the **diagonal** one and **hexagonal** matching fields for $\mathrm{Gr}(3,6)$ using OSCAR's polymake interface. By doing this we can observe that

$$\operatorname{vol}_{\mathbb{Z}}(\Pi_\Lambda) \;\le\; \deg \mathrm{Gr}(k,n),$$

with equality iff the Pluecker forms (maximal minors) are a Khovanskii (SAGBI) basis with respect to the weight matrix *inducing* the matching field or, in other words, $\Lambda$ gives a toric degeneration. The diagonal matching field has lattice volume $42$ and the hexagonal matching field $38$.

In [35]:
using Oscar

k, n = 3, 6;
triples = [[a, b, c] for a in 1:n for b in (a+1):n for c in (b+1):n];

## 1. Coherent matching fields from weight matrices

A weight matrix $M \in \mathbb{R}^{k \times n}$ induces the **matching field** $L$ (a function that assigns an ordering to each $k$-subset of $[n]$) if for every
$I \in \binom{[n]}{k}$ the Pluecker form $P_I = \det(X_I)$ has a *unique* lowest-weight term
$x_{1,c_1} x_{2,c_2} \cdots x_{k,c_k}$ where $L(I) = (c_1, c_2, \dots, c_k)$.

In [36]:
# all orderings of a k-set (k! = 6 here, so brute force is fine)
orderings(v) = length(v) == 1 ? [v] :
    [[v[i]; p] for i in eachindex(v) for p in orderings(deleteat!(copy(v), i))]

"L(I) = the unique minimum-weight ordering of I with respect to the weight matrix M."
function matching_field(M)
    L = Dict{Vector{Int},Vector{Int}}()
    for I in triples
        cs = orderings(I)
        ws = [sum(M[s, c[s]] for s in 1:k) for c in cs]
        count(==(minimum(ws)), ws) == 1 || error("tie at $I: M induces no matching field")
        L[I] = cs[argmin(ws)]
    end
    return L
end

# Diagonal
M_diag = [ 0  0  0  0  0  0
           6  5  4  3  2  1
          11  9  7  5  3  1 ]

# Hexagonal (cf Mohammadi-Shaw)
M_hex  = [ 0  0  0  0  0  0
           6  1  5  9  2  7
           5  8  2  7  3  1 ]

L_diag, L_hex = matching_field(M_diag), matching_field(M_hex)

# printed as in the paper: "421" means the initial term x[1,4]*x[2,2]*x[3,1]
println("   I     L_diag   L_hex")
for I in triples
    println("  ", join(I), "     ", join(L_diag[I]), "      ", join(L_hex[I]))
end

   I     L_diag   L_hex
  123     123      123
  124     124      421
  125     125      125
  126     126      126
  134     134      413
  135     135      153
  136     136      136
  145     145      451
  146     146      416
  156     156      156
  234     234      423
  235     235      523
  236     236      326
  245     245      425
  246     246      426
  256     256      526
  345     345      453
  346     346      436
  356     356      356
  456     456      456


`L_diag` is the identity on each triple but `L_hex` is different. The hexagonal matching field gets its name from the associated *Tropical Hyperplane Arrangement* whose dual subdivision contains a hexagonal hole. This feature gives an *additional generator* of the matching field ideal $P_{523}P_{416} - P_{526}P_{413} \in \mathcal{I}_\Lambda \setminus \operatorname{in}_{w_M}(\mathcal{I}_{3,6})$, with 36 minimal generators (35 is usual).

## 2. The matching field polytope

The matching field polytope is the convex hull of the exponents of the leading terms of the Pluecker forms:

$$\Pi_\Lambda \;=\; \operatorname{conv}\bigl\{\, v_{I,\Lambda} \;:\; I \in \tbinom{[n]}{k} \,\bigr\},$$

where $v_{I,\Lambda}$ is the exponent vector of $x_{I, \Lambda} := x_{1,c_1} x_{2,c_2} \cdots x_{k,c_k}$ where $L(I) = (c_1, c_2, \dots, c_k)$. For coherent
$\Lambda$ this is the polytope of the toric variety of the matching field ideal
$\mathcal{I}_\Lambda = \ker \phi_\Lambda$, where $\phi_\Lambda : \mathbb C[P_I]_I \rightarrow \mathbb C[x_{i,j}]_{i,j},\, P_I \mapsto x_{I, \Lambda}$.

In [10]:
function mf_polytope(L)
    V = zeros(Int, length(triples), k * n)          # rows = the 20 points v_{I,Λ}
    for (r, I) in enumerate(triples), i in 1:k
        V[r, (i - 1) * n + L[I][i]] = 1
    end
    return convex_hull(V)
end

P_diag, P_hex = mf_polytope(L_diag), mf_polytope(L_hex)

for (name, P) in (("diagonal", P_diag), ("hexagonal", P_hex))
    println(rpad(name, 11),
            "dim = ",         dim(P),
            ",  vertices = ", n_vertices(P),
            ",  lattice = ",  is_lattice_polytope(P),
            ",  full-dim = ", is_fulldimensional(P),
            ",  lattice-vol = ", lattice_volume(P)
    )
end

diagonal   dim = 9,  vertices = 20,  lattice = true,  full-dim = false,  lattice-vol = 42
hexagonal  dim = 9,  vertices = 20,  lattice = true,  full-dim = false,  lattice-vol = 38


In [12]:
println("degree of Grassmannian = ", degree(grassmann_pluecker_ideal(k, n)))

degree of Grassmannian = 6006


In [34]:
k, n = 3, 11;
triples = [[a, b, c] for a in 1:n for b in (a+1):n for c in (b+1):n];

M_diag = matrix([[(n-i)*(j-1) for i in 1:n] for j in 1:k])

L_diag = matching_field(M_diag)
P = mf_polytope(L_diag)

println("dim = ",         dim(P),
        ",  vertices = ", n_vertices(P),
        ",  lattice-vol = ", lattice_volume(P),
        ",  f-vector = ", f_vector(P)
    )

dim = 24,  vertices = 165,  lattice-vol = 23371634
